# Goal: Fuzzy match TX muni names between Mergent and TX election + debt issuance data

# Step 1: Import packages

In [1]:
#import packages
import os
from os import listdir
from os.path import isfile, join

import re
import numpy as np
import pandas as pd
import cython

import time
import datetime
import operator

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from scipy.sparse import csr_matrix

In [2]:
#https://github.com/ing-bank/sparse_dot_topn/issues/84
!pip install cython==0.29.36 numpy
!pip install sparse_dot_topn==0.3.0 --no-cache-dir --no-build-isolation
!pip install sparse-dot-topn-for-blocks --no-cache-dir --no-build-isolation
#!pip install string-grouper

  Preparing metadata (pyproject.toml) ... done
  Created wheel for sparse_dot_topn: filename=sparse_dot_topn-0.3.0-cp310-cp310-linux_x86_64.whl size=901790 sha256=38086c74bfad4f63ad8f7edc8dba0978b44fe240d09cdf66bd6a14e4499feefd
  Stored in directory: /tmp/pip-ephem-wheel-cache-ytv7jbus/wheels/64/82/0f/f024ac71b974553a29cd06a59e4314f6ec2a6b5f446572d2bd
Successfully built sparse_dot_topn
  Preparing metadata (pyproject.toml) ... done
  Created wheel for sparse-dot-topn-for-blocks: filename=sparse_dot_topn_for_blocks-0.3.1.post3-cp310-cp310-linux_x86_64.whl size=2445329 sha256=67dd1b83402bc651b9f8f6c1f2749afbb0425ad73608e582b0c76ba5de6b646c
  Stored in directory: /tmp/pip-ephem-wheel-cache-v3crbrs0/wheels/d8/68/ef/9f708a6e1a47f2b0ff13429390edf8d222759e6496050a0a58
Successfully built sparse-dot-topn-for-blocks


In [3]:
#https://github.com/Bergvca/string_grouper/blob/master/README.md
#https://bergvca.github.io/2017/10/14/super-fast-string-matching.html
!pip install string-grouper

  Preparing metadata (setup.py) ... done
  Created wheel for topn: filename=topn-0.0.7-cp310-cp310-linux_x86_64.whl size=1630901 sha256=3b4cf9b76250e2e43e8f5e2b30ceb2c7ca6324cf14b854ea31dce1183f3c3cc0
  Stored in directory: /root/.cache/pip/wheels/8d/84/d0/28aa7a3ede0cd09f4d3861c66db0a3dfd1dc50503f42eaf70e
Successfully built topn


In [4]:
from string_grouper import match_strings, match_most_similar, group_similar_strings, compute_pairwise_similarities, StringGrouper

# Step 2: Import the two datasets to match


In [5]:
# Mount Google Drive to this Notebook instance.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
#Load muni names from Mergent
df_mergent = pd.read_csv("./drive/MyDrive/Other research/TX_mergent_uniquegovt_formatch.csv", encoding="ISO-8859-1")
#make names uppercase
#df_gjf['parent_name_upper'] = df_gjf['parent_name'].str.upper()
#drop if company name missing
#df_gjf = df_gjf.dropna()
#print random 10 rows of data
print_mergent = df_mergent.sample(10)
print_mergent

,issuer_id,issuer_formatch,issuer_long_name,min_yr
347,2166,MIDLOTHIAN,MIDLOTHIAN TEX,2000
438,2761,ROYSE CITY,ROYSE CITY TEX,2005
16,77,ANNA,ANNA TEX,2017
254,1736,ITALY,ITALY TEX,2007
292,1889,LAKE WORTH,LAKE WORTH TEX,2009
220,1568,HARRISON CNTY,HARRISON CNTY TEX,2002
258,1742,JACKSBORO,JACKSBORO TEX,2015
60,277,BRAZORIA CNTY,BRAZORIA CNTY TEX,2006
114,593,COMAL CNTY,COMAL CNTY TEX,2002
243,1678,HOWARD CNTY,HOWARD CNTY TEX,2004


In [7]:
#How many munis?
print('Number of munis in Mergent data: {:,}\n'.format(df_mergent.shape[0]))
#555

Number of munis in Mergent data: 555



In [8]:
#import muni names from TX bond review board election + issuance data
df_brb = pd.read_csv("./drive/MyDrive/Other research/240512_TXBRB_uniquegovt_formatch.csv")

In [9]:
#make uppercase
df_brb['muni_upper'] = df_brb['muni_formatch'].str.upper()

In [10]:
#print random 10 rows
print_brb = df_brb.sample(10)
print_brb

,muni_formatch,muni_id_formatch,govtype,_merge,muni_upper
274,Iowa Park,419,CITY,only in using data,IOWA PARK
724,Georgetown,320,CITY,both in master and using data,GEORGETOWN
683,Daisetta,218,CITY,both in master and using data,DAISETTA
191,Falfurrias,281,CITY,only in using data,FALFURRIAS
78,Bovina,92,CITY,only in using data,BOVINA
246,Helotes,372,CITY,only in using data,HELOTES
925,Wills Point,911,CITY,both in master and using data,WILLS POINT
825,Ovilla,636,CITY,both in master and using data,OVILLA
390,Muleshoe,599,CITY,only in using data,MULESHOE
918,Westworth Village,894,CITY,both in master and using data,WESTWORTH VILLAGE


# Step 3: Do fuzzymatch

In [13]:
# Create all matches with at least 0.5 similarity
matches = match_strings(df_mergent['issuer_formatch'], df_brb['muni_upper'], min_similarity=0.5)
matches

,left_index,left_issuer_formatch,similarity,right_muni_upper,right_index
0,7,ALPINE,1.000000,ALPINE,0
1,61,BRAZOS CNTY,0.672480,BRAZOS COUNTRY,2
2,79,CALHOUN CNTY,0.847751,CALHOUN COUNTY,3
3,118,COOKE CNTY,0.781397,COOKE COUNTY,5
4,165,FAIRVIEW,1.000000,FAIRVIEW,7
...,...,...,...,...,...
771,543,WINNSBORO,1.000000,WINNSBORO,931
772,547,WOODVILLE,1.000000,WOODVILLE,933
773,549,WYLIE,1.000000,WYLIE,934
774,552,YOUNG CNTY,0.723461,YOUNG COUNTY,935


In [14]:
#Export to csv
matches.to_csv("./drive/MyDrive/Other research/240512_TX_uniquegovt_fuzzymatchresults.csv")